In [1]:
import pandas as pd

# Load master dataset
master = pd.read_csv("MASTER_HAZARD_DATASET.csv")

# Convert timestamp back to datetime
master["timestamp"] = pd.to_datetime(master["timestamp"], utc=True, errors="coerce")

# Identify hazard events (Flic button)
hazard_events = master[master["id_flic"].notna()].copy()

print("Hazard events detected:", len(hazard_events))

# Remove duplicates
hazard_events = hazard_events.drop_duplicates(subset=["id_flic", "ts_flic"])

# Window duration
before = pd.Timedelta(seconds=30)
after = pd.Timedelta(seconds=10)

windows = []

for _, event in hazard_events.iterrows():
    t = event["timestamp"]  # now it is datetime
    start = t - before
    end = t + after
    
    window_slice = master[
        (master["timestamp"] >= start) &
        (master["timestamp"] <= end)
    ].copy()
    
    window_slice["hazard_time"] = t
    window_slice["hazard_id"] = event["id_flic"]
    
    windows.append(window_slice)

hazard_dataset = pd.concat(windows, ignore_index=True)

# Save
hazard_dataset.to_csv("HAZARD_EVENT_WINDOWS.csv", index=False)

print("Final hazard window dataset shape:", hazard_dataset.shape)

C:\Users\toufi\AppData\Local\Temp\ipykernel_19624\2892626001.py:4: DtypeWarning: Columns (0: ts_flic, 1: aplicom_flic, 2: address_flic, 3: geom_flic, 4: timestamp, 5: source_flic, 6: ts_g, 7: aplicom_g, 8: address_g, 9: advdata_g, 10: geom_g, 11: source_g) have mixed types. Specify dtype option on import or set low_memory=False.
  master = pd.read_csv("MASTER_HAZARD_DATASET.csv")


Hazard events detected: 8515
Final hazard window dataset shape: (57718, 82)
